# Benchmark JAX GPU Colab — PR #14d (Sprint 7.x — Performance)

**Objetivo**: validar que o ``forward_pure_jax`` otimizado (bucketing + jit + vmap) funciona eficientemente em GPU T4/A100 do Colab Pro+.

**Resultados em CPU macOS Intel Core i9 (pós-otimização Sprint 7.x):**

| Modelo | Antes (ms/mod) | **Depois (ms/mod)** | Speedup |
|:-------|---------------:|-------------------:|--------:|
| oklahoma_3 | 17.135 | **15,3** | **1.123×** |
| oklahoma_5 | 17.111 | **23,9** | **717×** |
| oklahoma_28 | 17.359 | **113,2** | **153×** |

Alvo de GPU T4: < **100 ms/modelo oklahoma_3** + VRAM < **4 GB** (contra 58,8 s/modelo + 12 GB VRAM do notebook anterior).

**Este notebook deve ser executado manualmente no Colab Pro+ com runtime T4 ou A100.**

## 0. Setup Colab + JAX CUDA

In [ ]:
!pip install -q numpy 'numba>=0.60' 'jax[cuda12]>=0.4.30'
!git clone -b main https://github.com/daniel-guitarplayer-8/geosteering-ai.git /content/Geosteering_AI
%cd /content/Geosteering_AI

In [ ]:
import os, sys, time
os.environ['JAX_ENABLE_X64'] = 'True'
sys.path.insert(0, '/content/Geosteering_AI')

import jax
jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices())
print('JAX default backend:', jax.default_backend())
print('JAX version:', jax.__version__)

## 1. Forward JAX nativo OTIMIZADO — modelos canônicos

Agora com bucketing + `jax.jit` cached + `jax.vmap` duplo interno.

In [ ]:
import numpy as np
from geosteering_ai.simulation._jax.forward_pure import (
    build_static_context, forward_pure_jax,
)
from geosteering_ai.simulation.validation.canonical_models import get_canonical_model

def bench_forward_jax_native(model_name: str, n_pos: int = 100, n_iter: int = 5):
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, n_pos)
    ctx = build_static_context(
        m.rho_h, m.rho_v, m.esp, positions_z,
        np.array([20000.0]), 1.0, 0.0,
    )
    # Warmup JIT (primeiro trace é lento)
    t0 = time.perf_counter()
    H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()
    t_first = time.perf_counter() - t0
    t0 = time.perf_counter()
    for _ in range(n_iter):
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
        H.block_until_ready()
    elapsed = (time.perf_counter() - t0) / n_iter
    return t_first, elapsed, np.asarray(H).shape

print(f'{"Modelo":<20} {"n_pos":>6} {"1a chamada (s)":>15} {"T_forward (ms)":>15} {"throughput mod/h":>18}')
for name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28', 'hou_7', 'viking_graben_10']:
    t_first, t_med, shape = bench_forward_jax_native(name, n_pos=100, n_iter=5)
    throughput = 3600.0 / t_med
    print(f'{name:<20} {shape[0]:>6d} {t_first:>15.2f} {t_med*1000:>15.2f} {throughput:>16.0f}')

## 2. Monitoramento VRAM GPU

In [ ]:
# Checa memória VRAM consumida pelo JAX (não deve ultrapassar ~4 GB pós-otimização)
if jax.default_backend() == 'gpu':
    device = jax.devices()[0]
    try:
        stats = device.memory_stats()
        bytes_in_use = stats.get('bytes_in_use', 0)
        print(f'VRAM em uso: {bytes_in_use / (1024**3):.2f} GB')
        print(f'Peak VRAM:   {stats.get("peak_bytes_in_use", 0) / (1024**3):.2f} GB')
    except Exception as e:
        print(f'memory_stats não disponível: {e}')
else:
    print('Executando em CPU; sem VRAM para monitorar')

## 3. Jacobiano jax.jacfwd em GPU

In [ ]:
def bench_jacfwd_gpu(model_name: str, n_pos: int = 50, n_iter: int = 3):
    m = get_canonical_model(model_name)
    positions_z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, n_pos)
    ctx = build_static_context(
        m.rho_h, m.rho_v, m.esp, positions_z,
        np.array([20000.0]), 1.0, 0.0,
    )
    def fwd(rh, rv):
        return forward_pure_jax(rh, rv, ctx)
    # Warmup
    J_h, J_v = jax.jacfwd(fwd, argnums=(0, 1))(ctx.rho_h_jnp, ctx.rho_v_jnp)
    J_h.block_until_ready()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        J_h, J_v = jax.jacfwd(fwd, argnums=(0, 1))(ctx.rho_h_jnp, ctx.rho_v_jnp)
        J_h.block_until_ready()
    elapsed = (time.perf_counter() - t0) / n_iter
    return elapsed, J_h.shape

print(f'{"Modelo":<20} {"n_layers":>8} {"T_jacfwd (ms)":>16}')
for name in ['oklahoma_3', 'oklahoma_5', 'hou_7']:
    t, shape = bench_jacfwd_gpu(name, n_pos=50, n_iter=3)
    print(f'{name:<20} {shape[-1]:>8d} {t*1000:>15.2f}')

## 4. Tabela final — preencher após executar

| Modelo | CPU Intel i9 pós-otim (ms) | GPU T4 pós-otim (ms) | VRAM (GB) | Speedup GPU/CPU |
|:-------|--------------------------:|--------------------:|----------:|----------------:|
| oklahoma_3 | 15,3 | (executar §1) | (executar §2) | (calcular) |
| oklahoma_5 | 23,9 | (executar §1) | (executar §2) | (calcular) |
| oklahoma_28 | 113,2 | (executar §1) | (executar §2) | (calcular) |